In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:90%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

# RAG 절차
- https://law.go.kr/법령/소득세법 에서 doc파일로 다운로드 (hwp는 파이썬이 못 읽음, pdf는 한글의 문장,문단,단어 인식이 불가해 짤리는 경우가 많음)
    - 다운로드 후 docx로 변경(새로 저장하기)
   
### [ RAG 구현 절차 ]


```
1.문서의 내용을 읽는다(document_loader를 이용)
(1)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/ 
(2)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/microsoft_word/
%pip install --upgrade --quiet  docx2txt

2.	문서를 쪼갠다(한번에 이해하고 처리할 수 있는 입력+출력 토큰수가 제한)
(1)	 https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/#splitting-text-from-languages-without-word-boundaries 
%pip install -qU langchain-text-splitters
3.	쪼갠 문서를 임베딩하여 vector database에 넣음
(1)	OpenAIEmbeddings나 UpstageEmbeddings이용해서 임베딩
(2)	https://python.langchain.com/v0.2/docs/integrations/vectorstores/chroma/  
%pip install –q langchain-chroma
4.	질문을 이용해 유사도 검색
5.	유사도 검색한 문서를 LLM에 질문으로 전달하여 답변 얻음(제공되는 Prompt활용)
(1)	https://python.langchain.com/v0.2/docs/tutorials/rag/
%pip install –q langchain langchainhub

```

# 0. 패키지 설치 

In [2]:
# 문서 읽어오기
%pip install --upgrade --quiet  docx2txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
# 문서 분리하기 : 텍스트를 청크로 나누는 기능만 있는 경량 모듈
%pip install -qU langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [4]:
# 백터DB(로컬DB)
%pip install -q langchain-chroma

Note: you may need to restart the kernel to use updated packages.


In [5]:
# 제공되는 prompt 사용
%pip install -q langchain langchainhub

Note: you may need to restart the kernel to use updated packages.


# 1. 문서 불러오기(비추)

In [22]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
document = loader.load()  # loder -> loader로 수정
# document

CPU times: total: 4.42 s
Wall time: 4.44 s


In [7]:
len(document)

1

In [8]:
document[0].page_content[:100]

'소득세법\n\n소득세법\n\n[시행 2025. 7. 1.] [법률 제20615호, 2024. 12. 31., 일부개정]\n\n기획재정부(재산세제과(양도소득세)) 044-215-4312\n\n기획'

# 2. 문서를 분리하면서 불러오기(추천)

In [1]:
import time
start =  time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 문자 단위로 분리 / 문자 분리 기준 : 문자수 (토큰이 아님) 

loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,    # 문서를 분리할 때 1500글자씩 분리
    chunk_overlap=200,  # 200글자정도 겹쳐서 분리하기
    
    )
# 1번째 chunk 1~1,450글자 
# 2번째 chunk 1250~ 2750글자 
document= loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print(f"문서 분리 시간 : 총 {runtime}")
# print(document)

문서 분리 시간 : 총 4.384401798248291


In [10]:
# chunk 갯수
len(document)

183

In [11]:
document[0].page_content

'소득세법\n\n소득세법\n\n[시행 2025. 7. 1.] [법률 제20615호, 2024. 12. 31., 일부개정]\n\n기획재정부(재산세제과(양도소득세)) 044-215-4312\n\n기획재정부(소득세제과(근로소득)) 044-215-4216\n\n기획재정부(금융세제과(이자소득, 배당소득)) 044-215-4233\n\n기획재정부(소득세제과(사업소득, 기타소득)) 044-215-4217\n\n\n\n제1장 총칙 <개정 2009. 12. 31.>\n\n\n\n제1조(목적) 이 법은 개인의 소득에 대하여 소득의 성격과 납세자의 부담능력 등에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지함을 목적으로 한다.\n\n[본조신설 2009. 12. 31.]\n\n[종전 제1조는 제2조로 이동 <2009. 12. 31.>]\n\n\n\n제1조의2(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2010. 12. 27., 2014. 12. 23., 2018. 12. 31.>\n\n1. “거주자”란 국내에 주소를 두거나 183일 이상의 거소(居所)를 둔 개인을 말한다.\n\n2. “비거주자”란 거주자가 아닌 개인을 말한다.\n\n3. “내국법인”이란 「법인세법」 제2조제1호에 따른 내국법인을 말한다.\n\n4. “외국법인”이란 「법인세법」 제2조제3호에 따른 외국법인을 말한다.\n\n5. “사업자”란 사업소득이 있는 거주자를 말한다.\n\n② 제1항에 따른 주소ㆍ거소와 거주자ㆍ비거주자의 구분은 대통령령으로 정한다.\n\n[본조신설 2009. 12. 31.]\n\n\n\n제2조(납세의무) ① 다음 각 호의 어느 하나에 해당하는 개인은 이 법에 따라 각자의 소득에 대한 소득세를 납부할 의무를 진다.\n\n1. 거주자\n\n2. 비거주자로서 국내원천소득(國內源泉所得)이 있는 개인\n\n② 다음 각 호의 어느 하나에 해당하는 자는 이 법에 따라 원천징수한 소득세를 납부할 의무를 진다.\n\n1. 거주자\n\n2. 비거주자\n

In [12]:
len(document[0].page_content)

1464

In [13]:
# chunk의 글자수
# [len(doc.page_content) for doc in document]
print(max(len(doc.page_content) for doc in document))
print(min(len(doc.page_content) for doc in document))

1497
1055


# 3. 분리된 문서를 임베딩 → 벡터 데이터베이스 저장
- 임베딩 모델 : openAI API의 text-embedding-3-large (기본:text-embedding-ada-002)
- 백터 데이터 베이스 : chroma

In [2]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(
    model="text-embedding-3-large")

In [15]:
embeddings = embedding.embed_documents(
    [
        "소득세법 ",
        document[0].page_content
    ]
)


In [16]:
len(embeddings), len(embeddings[0]), len(embeddings[1])

(2, 3072, 3072)

In [17]:
# print(document)

In [3]:
%%time
from langchain_chroma import Chroma

# 데이터를 처음 저장할 때 방식
# database = Chroma.from_documents(
#     documents=document,
#     embedding=embedding,
#     collection_name='tex-collection', # 생략시 이름 랜덤 생성
#     persist_directory= './chroma'     # 생략시 로컬데이터베이스에 저장안됨. 프로그램 종료시 db 날아감
# )


# 이미 저장된 vector DB를 사용할 때
database = Chroma(
    embedding_function=embedding,
    collection_name="tex-collection",
    persist_directory='./chroma'
)

CPU times: total: 1.83 s
Wall time: 8.34 s


# 4. Vector DB에 질문과 유사도 검색(답변 생성을 위한 retrieval)

In [25]:
database

In [4]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query,
                                           k=3) # 기본 k는 4

In [5]:
retrieved_docs

[Document(id='6d239116-846d-4065-be61-3fb39af9cd1f', metadata={'source': './tax_docs/소득세법(법률)(제20615호)(20250701).docx'}, page_content='[전문개정 2009. 12. 31.]\n\n\n\n제10조(납세지의 변경신고) 거주자나 비거주자는 제6조부터 제9조까지의 규정에 따른 납세지가 변경된 경우 변경된 날부터 15일 이내에 대통령령으로 정하는 바에 따라 그 변경 후의 납세지 관할 세무서장에게 신고하여야 한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제11조(과세 관할) 소득세는 제6조부터 제10조까지의 규정에 따른 납세지를 관할하는 세무서장 또는 지방국세청장이 과세한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제2장 거주자의 종합소득 및 퇴직소득에 대한 납세의무 <개정 2009. 12. 31.>\n\n\n\n제1절 비과세 <개정 2009. 12. 31.>\n\n\n\n제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31.>\n\n1. 「공익신탁법」에 따른 공익신탁의 이익\n\n2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득\n\n가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득\n\n나. 1개의 주택을 소유하는 자의 주택임대소득(

In [6]:
retrieved_docs[0].page_content

'[전문개정 2009. 12. 31.]\n\n\n\n제10조(납세지의 변경신고) 거주자나 비거주자는 제6조부터 제9조까지의 규정에 따른 납세지가 변경된 경우 변경된 날부터 15일 이내에 대통령령으로 정하는 바에 따라 그 변경 후의 납세지 관할 세무서장에게 신고하여야 한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제11조(과세 관할) 소득세는 제6조부터 제10조까지의 규정에 따른 납세지를 관할하는 세무서장 또는 지방국세청장이 과세한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제2장 거주자의 종합소득 및 퇴직소득에 대한 납세의무 <개정 2009. 12. 31.>\n\n\n\n제1절 비과세 <개정 2009. 12. 31.>\n\n\n\n제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31.>\n\n1. 「공익신탁법」에 따른 공익신탁의 이익\n\n2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득\n\n가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득\n\n나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변 생성

In [7]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano') # llm 객체 생성

In [8]:
prompt = f"""[idnetity]
- 당신은 최고의 한국 소득세 전문가입니다.
- [context]를 참고해서 사용자의 질문에 답변해 주세요.
[context]는 다음과 같습니다.
{retrieved_docs}
Question:{query}"""

In [9]:
ai_message = llm.invoke(prompt)

In [10]:
print(ai_message.content)

연봉 5,000만원인 직장인의 소득세를 계산하려면, 과세표준과 누진세율을 적용하여야 합니다. 아래는 일반적인 계산 과정입니다.

1. 근로소득 기본공제액 적용  
- 근로소득자에게는 기본공제 150만원이 적용됩니다.  
- 연봉: 50,000,000원  
- 공제 후 과세표준: 50,000,000원 - 1,500,000원 = 48,500,000원

2. 근로소득세 누진세율 적용  
대한민국 소득세 법령에 따른 2023년 기준 세율은 다음과 같습니다.  
| 과세표준 구간 | 세율 | 누진공제액 | 산출세액 |  
|----------------|--------|----------------|--------------|  
| 1,200만원 이하 | 6% | 0 | 과세표준 × 6% |  
| 1,200만원 초과 ~ 4,600만원 이하 | 15% | 108만원 | (과세표준 × 15%) - 108만원 |  
| 4,600만원 초과 ~ 8,800만원 이하 | 24% | 522만원 | (과세표준 × 24%) - 522만원 |  

※ 참고로, 연봉에서 공제된 과세표준이 48,500,000원임을 고려하여 세금을 계산한다.  

3. 계산:  
- 과세표준: 48,500,000원

- 1,200만원 이하 부분: 12,000,000원 × 6% = 720,000원  
- 다음 3,400만원 (4,600만원 - 1,200만원) 부분: (48,500,000원 - 12,000,000원) = 36,500,000원  
  - 세율 15% 적용: 36,500,000원 × 15% = 5,475,000원  
  - 공제액: 1,080,000원 (누진공제액) 적용 안 함, 세율만 고려하면 됨

- 최종 세액:  
  720,000원 + (36,500,000원 × 15% - 1,080,000원) =  
  720,000원 + (5,475,000원 - 1,080,000원) =  
  720,000원 + 4,395,000원 = 5,115,000원

4. 결론:  
이 계산은 기본적인 공제만 반

# 6. Augmentation을 위한 제공되는 Prompt활용하여  langchain으로 답변 생성

In [11]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"

from langchain import hub
prompt = hub.pull("rlm/rag-prompt")
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

### RetrievalQA를 통해 LLM 전달(create_retrieval_chain이 대체)
```
Query → retrieval전달(백터 검색 수행) → retrieval 문서 → prompt의 {context}에 삽입 → 전달받은 query → prompt의 {question}에 삽입
```

In [14]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = database.as_retriever(search_kwargs={'k':5}),
    chain_type_kwargs={"prompt":prompt}
)

In [15]:
ai_message = qa_chain.invoke({"query":query})

In [16]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5천만원인 직장인의 소득세는 구체적인 세율과 공제액에 따라 달라지기 때문에 정확한 금액을 알기 어렵습니다. 그러나 일반적으로 일정 금액 이상의 소득에는 누진세율이 적용되며, 공제 후 과세표준에 따라 세액이 산출됩니다. 자세한 계산을 위해서는 세법상 공제액과 세율표를 참고하는 것이 필요합니다.'}